# Absolute disparity: prediction errors before rounding

Compare the saved allocations from $z_i=h_i+\beta(\mu_i-h_i)$ rounded once to grid step $\delta$.
**All disparity is evaluated using true needs**, regardless of the needs supplied to the DP.

The plot shows mean absolute disparity against beta, one curve per delta, and light **mean ± one
sample standard deviation** bands with lower bounds clipped at zero. LDA-exp, LDA-tru, and OPT are
horizontal references. The bands describe variation across model/cohort allocations, not confidence intervals.
Deduplicated selections are expanded to one observation per eligible model, matching the other results notebooks.

This notebook reads saved results only; it does not run allocation algorithms. It can display an incomplete
sweep: only completed, verified points are plotted. Rerun all cells to include newly completed runs.

A second figure fixes β = 1 and plots disparity against δ, isolating the effect of grid spacing
when the full prediction error is retained. Smaller δ means finer discretization.


In [2]:
import hashlib
import json
from ast import literal_eval
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "src/allocations.py").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the research repository or its test/ folder.")
ALLOCATION_DIR = ROOT / "data/results/allocations"
EXPERIMENTS_ROOT = ALLOCATION_DIR / "error_before_rounding"

# None selects the configuration with the most recently completed run.
# To pin a configuration, set its directory name (shown in the table below) or absolute path.
EXPERIMENT_DIR = None
BASELINE_PEOPLE_PATH = None  # Optional override for legacy baseline cohort metadata.
BASELINE_FILES = {}          # Optional overrides, e.g. {"OPT": Path("...")}.
SHOW_REFERENCE_BANDS = True
SAVE_OUTPUTS = True


## Choose a saved experiment

Sample size, model count, utility tolerance, input files, and other settings come from the saved
configuration; different configurations are never pooled. Directories with no completed runs are
listed but are not automatically selected. Completion metadata and file hashes are checked before loading.


In [3]:
def file_hash(path):
    with Path(path).open("rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()

available = []
for manifest_path in EXPERIMENTS_ROOT.glob("*/config.json"):
    saved_config = json.loads(manifest_path.read_text())
    markers = list(manifest_path.parent.glob("run_beta-*_delta-*.json"))
    available.append({
        "directory": manifest_path.parent.name, "n": saved_config["sample_size"],
        "k": saved_config["k"], "models": saved_config["n_models"],
        "cohorts_per_seed": saved_config["iters_per_seed"], "completed_runs": len(markers),
        "latest_completion": max((p.stat().st_mtime for p in markers), default=0),
    })
if not available:
    raise FileNotFoundError("No experiment configurations found. Run 02-experiments_error_before_rounding.ipynb first.")
available_configs = pd.DataFrame(available).sort_values("latest_completion", ascending=False)
display(available_configs.drop(columns="latest_completion"))
if EXPERIMENT_DIR is None:
    completed = available_configs.loc[available_configs.completed_runs.gt(0)]
    if completed.empty:
        raise RuntimeError("No beta/grid combination has completed yet. Rerun after the first completion marker is saved.")
    EXPERIMENT_DIR = EXPERIMENTS_ROOT / completed.iloc[0].directory
else:
    EXPERIMENT_DIR = Path(EXPERIMENT_DIR)
    if not EXPERIMENT_DIR.is_absolute():
        EXPERIMENT_DIR = EXPERIMENTS_ROOT / EXPERIMENT_DIR
config = json.loads((EXPERIMENT_DIR / "config.json").read_text())
if config["experiment"] != "error_before_rounding" or config["shrinkage_order"] != "before_rounding" or config["alpha"] != 0:
    raise ValueError("Choose a before-rounding experiment with alpha=0.")
config_id = hashlib.sha256(json.dumps(config, sort_keys=True, indent=2).encode()).hexdigest()[:12]
run_files = sorted(EXPERIMENT_DIR.glob("run_beta-*_delta-*.json"))
if not run_files:
    raise RuntimeError("The selected configuration has no completed runs yet.")
print("Selected:", EXPERIMENT_DIR.name)
print(f"n={config['sample_size']}, k={config['k']}, selection rate={config['selection_rate']:g}; "
      f"{len(run_files)} completed beta/grid combinations")


,directory,n,k,models,cohorts_per_seed,completed_runs
1,size-100_models-100_iters-5_7807aca5c013,100,10,100,5,15
0,size-250_models-100_iters-10_1b191eed0a85,250,25,100,10,0


Selected: size-100_models-100_iters-5_7807aca5c013
n=100, k=10, selection rate=0.1; 15 completed beta/grid combinations


In [4]:
def input_path(saved_name, fallback):
    path = Path(saved_name)
    return path if path.exists() else fallback

prediction_path = input_path(config["prediction_path"], ROOT / "data/predictions" / Path(config["prediction_path"]).name)
raw_path = input_path(config["raw_data_path"], ROOT / "data/obermeyer" / Path(config["raw_data_path"]).name)
for path, expected in ((prediction_path, config["prediction_sha256"]), (raw_path, config["raw_data_sha256"])):
    if file_hash(path) != expected:
        raise ValueError(f"Input changed since the experiment was run: {path}")

NEED_COL, GROUP_COL = config["need_column"], config["group_column"]
model_names = [f"m_{i}" for i in range(1, config["n_models"] + 1)]
predictions = pd.read_csv(prediction_path, usecols=["person_id", "seed", "y", NEED_COL, *model_names])
groups = pd.read_csv(raw_path, usecols=["person_id", GROUP_COL])
predictions = predictions.merge(groups, on="person_id", how="left", validate="many_to_one")
expected_keys = set()
eligible_by_seed = {}
split_by_seed = {}
for seed in config["seeds"]:
    losses = predictions.loc[(predictions.seed == seed) & predictions.person_id.eq(-1), model_names]
    if len(losses) != 1 or not np.isfinite(losses.to_numpy(dtype=float)).all():
        raise ValueError(f"Expected one finite validation-loss row for seed {seed}.")
    losses = losses.iloc[0]
    models = losses.index[losses <= losses.min() + config["rashomon_epsilon"]].tolist()
    eligible_by_seed[seed] = set(models)
    expected_keys.update((seed, i, model) for i in range(config["iters_per_seed"]) for model in models)
    split = predictions.loc[(predictions.seed == seed) & predictions.person_id.ge(0)].set_index("person_id")
    if not split.index.is_unique:
        raise ValueError(f"Duplicate person IDs in seed {seed}.")
    split_by_seed[seed] = split

def read_people(path):
    people = pd.read_csv(path, converters={"people": literal_eval})
    if people.duplicated(["seed", "iteration"]).any():
        raise ValueError(f"Duplicate cohorts in {path}")
    return people

def cohort_map(people):
    wanted = {(seed, i) for seed in config["seeds"] for i in range(config["iters_per_seed"])}
    found = {(int(r.seed), int(r.iteration)): tuple(r.people) for r in people.itertuples(index=False)}
    if not wanted <= found.keys():
        raise ValueError("Missing cohorts for this experiment configuration.")
    found = {key: found[key] for key in sorted(wanted)}
    for ids in found.values():
        if len(ids) != config["sample_size"] or len(set(ids)) != len(ids):
            raise ValueError("A cohort has the wrong sample size or duplicate people.")
    return found

def read_allocations(path):
    return pd.read_csv(path, converters={"selected": literal_eval, "models": literal_eval})

def evaluate(allocations, cohorts, *, algorithm, beta=np.nan, delta=np.nan, strict=True):
    """Expand deduplicated selections and evaluate group means using true needs."""
    rows = []
    seen = set()
    for row in allocations.itertuples(index=False):
        cohort_key = (int(row.seed), int(row.iteration))
        if cohort_key not in cohorts:
            if strict:
                raise ValueError("An allocation belongs to an unexpected cohort.")
            continue
        selected_ids = list(row.selected)
        if len(selected_ids) != config["k"] or len(set(selected_ids)) != config["k"]:
            raise ValueError("An allocation has the wrong number of selected people.")
        if not set(selected_ids) <= set(cohorts[cohort_key]):
            raise ValueError("An allocation contains people outside its recorded cohort.")
        selected = split_by_seed[cohort_key[0]].loc[selected_ids]
        if not selected[GROUP_COL].isin([0, 1]).all() or not np.isfinite(selected[NEED_COL]).all():
            raise ValueError("Selected people have invalid group labels or true needs.")
        h0 = selected.loc[selected[GROUP_COL].eq(0), NEED_COL]
        h1 = selected.loc[selected[GROUP_COL].eq(1), NEED_COL]
        disparity = float(abs(h1.mean() - h0.mean())) if len(h0) and len(h1) else np.nan
        if not row.models:
            raise ValueError("An allocation row has no associated models.")
        for model in row.models:
            key = (*cohort_key, model)
            if key not in expected_keys:
                if strict:
                    raise ValueError(f"Unexpected model/cohort: {key}")
                continue
            if key in seen:
                raise ValueError(f"Multiple allocations for the same model/cohort: {key}")
            seen.add(key)
            rows.append({
                "algorithm": algorithm, "beta": beta, "delta": delta,
                "seed": cohort_key[0], "iteration": cohort_key[1], "model": model,
                "n": config["sample_size"], "k": config["k"], "k_0": len(h0), "k_1": len(h1),
                "selected": tuple(sorted(selected_ids)), "absolute_disparity": disparity,
                "expected_utility": float(selected[model].mean()),
                "realized_utility": float(selected.y.mean()),
            })
    if seen != expected_keys:
        raise ValueError(f"Allocation coverage differs from the experiment: {len(expected_keys - seen)} model/cohort rows missing.")
    return pd.DataFrame(rows)

loaded_runs = {}
reference_cohorts = None
coverage = []
for marker in run_files:
    metadata = json.loads(marker.read_text())
    beta, delta = float(metadata["beta"]), float(metadata["need_step"])
    if (metadata["config_id"] != config_id or metadata["shrinkage_order"] != "before_rounding"
            or metadata["alpha"] != 0 or not 0 <= beta <= 1 or not 0 < delta <= 1):
        raise ValueError(f"Run metadata does not match the configuration: {marker}")
    if (beta, delta) in loaded_runs:
        raise ValueError(f"Duplicate beta/delta run: {(beta, delta)}")
    suffix = marker.stem.removeprefix("run_")
    paths = {name: EXPERIMENT_DIR / f"{prefix}_{suffix}.csv"
             for name, prefix in (("allocations", "lda_exp"), ("people", "people"), ("diagnostics", "diagnostics"))}
    for name, path in paths.items():
        if not path.exists() or file_hash(path) != metadata["sha256"][name]:
            raise ValueError(f"Saved run is incomplete or has changed: {path}")
    people = read_people(paths["people"])
    cohorts = cohort_map(people)
    if reference_cohorts is None:
        reference_cohorts = cohorts
    elif cohorts != reference_cohorts:
        raise ValueError("Cohort identities/order differ between beta/grid combinations.")
    evaluated = evaluate(read_allocations(paths["allocations"]), cohorts,
                         algorithm="LDA-exp-before-rounding", beta=beta, delta=delta)
    if len(evaluated) != metadata["n_model_cohort_allocations"] or len(cohorts) != metadata["n_cohorts"]:
        raise ValueError("Run coverage does not agree with saved metadata.")
    loaded_runs[(beta, delta)] = evaluated
    coverage.append({"beta": beta, "delta": delta, "model_cohort_rows": len(evaluated)})
experiment_results = pd.concat(loaded_runs.values(), ignore_index=True)
display(pd.DataFrame(coverage).pivot(index="beta", columns="delta", values="model_cohort_rows").sort_index(axis=1, ascending=False))
print("Table entries count model/cohort allocations. Blank cells have not completed; no missing point is fabricated.")


delta,1.0,0.5,0.2
beta,,,
0.00,2490,2490,2490
0.25,2490,2490,2490
0.50,2490,2490,2490
0.75,2490,2490,2490
1.00,2490,2490,2490


Table entries count model/cohort allocations. Blank cells have not completed; no missing point is fabricated.


## Reference allocations

LDA-tru uses a saved beta=0 endpoint when available. LDA-exp uses beta=1, delta=1 when available;
this is the original whole-number predicted-needs baseline. Otherwise, the corresponding existing baseline
CSV is used. OPT comes from its existing baseline CSV. No reference algorithm is rerun.

Legacy reference files must contain the same people in each cohort and every eligible model in
this configuration. Their list order may differ because evaluating saved selections depends on membership;
the beta/grid experiment itself is checked for identical order. Extra legacy cohorts/models are filtered out.
Legacy filenames do not record every
setting, so use files produced with the matching utility tolerance; override their paths above if necessary.

If an allocation selects only one group, its disparity is undefined, not zero. Such rows remain in the
results, are counted below, and are excluded from the mean and standard deviation for that series.


In [5]:
tag = (f"{config['method']}_{config['model_class']}_{config['prediction_method']}"
       f"_y-qualified_gagne_{config['need_threshold']}_rate-{config['selection_rate']}_size-{config['sample_size']}")
legacy_people_path = Path(BASELINE_PEOPLE_PATH) if BASELINE_PEOPLE_PATH is not None else ALLOCATION_DIR / f"people_{tag}.csv"
reference_frames, reference_sources = [], []
for algorithm, prefix in (("LDA-exp", "lda_exp"), ("LDA-tru", "lda_tru"), ("OPT", "opt")):
    endpoint = None
    if algorithm == "LDA-exp" and (1.0, 1.0) in loaded_runs:
        endpoint = (1.0, 1.0)
    elif algorithm == "LDA-tru":
        zero_runs = sorted((key for key in loaded_runs if key[0] == 0), key=lambda key: -key[1])
        endpoint = zero_runs[0] if zero_runs else None
    if endpoint is not None:
        frame = loaded_runs[endpoint].copy()
        source = f"Experiment endpoint: beta={endpoint[0]:g}, delta={endpoint[1]:g}"
    else:
        path = Path(BASELINE_FILES.get(algorithm, ALLOCATION_DIR / f"{prefix}_{tag}.csv"))
        if not path.exists() or not legacy_people_path.exists():
            raise FileNotFoundError(f"Missing matching {algorithm} baseline: {path} or {legacy_people_path}. "
                                    "Supply matching saved reference files; this notebook does not run allocations.")
        legacy_cohorts = cohort_map(read_people(legacy_people_path))
        if any(set(legacy_cohorts[key]) != set(ids) for key, ids in reference_cohorts.items()):
            raise ValueError(f"{algorithm} baseline cohorts differ from the experiment. Choose matching saved baseline files.")
        frame = evaluate(read_allocations(path), reference_cohorts, algorithm=algorithm, strict=False)
        source = str(path)
    frame["algorithm"] = algorithm
    frame["beta"], frame["delta"] = np.nan, np.nan
    reference_frames.append(frame)
    reference_sources.append({"algorithm": algorithm, "source": source})
reference_results = pd.concat(reference_frames, ignore_index=True)
df_results = pd.concat([experiment_results, reference_results], ignore_index=True)
display(pd.DataFrame(reference_sources))

def summarize(frame, keys):
    stats = frame.groupby(keys, dropna=False).agg(
        mean=("absolute_disparity", "mean"), std=("absolute_disparity", "std"),
        n_valid=("absolute_disparity", "count"), n_total=("absolute_disparity", "size"),
        expected_utility=("expected_utility", "mean"), realized_utility=("realized_utility", "mean"),
    ).reset_index()
    stats["n_undefined"] = stats.n_total - stats.n_valid
    stats["lower"] = (stats["mean"] - stats["std"]).clip(lower=0)
    stats["upper"] = stats["mean"] + stats["std"]
    return stats

experiment_stats = summarize(experiment_results, ["beta", "delta"])
reference_stats = summarize(reference_results, ["algorithm"])
if reference_stats.n_valid.eq(0).any():
    raise ValueError("A reference has no defined disparity values, so its reference line cannot be plotted.")
display(experiment_stats.sort_values(["delta", "beta"], ascending=[False, True]))
display(reference_stats)
print("SD uses ddof=1 across model/cohort allocations; it is not a confidence interval.")


,algorithm,source
0,LDA-exp,"Experiment endpoint: beta=1, delta=1"
1,LDA-tru,"Experiment endpoint: beta=0, delta=1"
2,OPT,/Users/kk985/PhD/Research/LessDiscriminatoryAl...


,beta,delta,mean,std,n_valid,n_total,expected_utility,realized_utility,n_undefined,lower,upper
2,0.00,1.0,0.241053,0.723732,2490,2490,0.972981,0.963253,0,0.000000,0.964785
5,0.25,1.0,0.491150,0.699334,2490,2490,0.972546,0.964096,0,0.000000,1.190484
8,0.50,1.0,0.787663,0.754198,2490,2490,0.972037,0.967711,0,0.033465,1.541862
11,0.75,1.0,1.177647,0.896762,2490,2490,0.972105,0.963454,0,0.280884,2.074409
14,1.00,1.0,1.650181,1.151700,2490,2490,0.972401,0.963896,0,0.498481,2.801881
1,0.00,0.5,0.241053,0.723732,2490,2490,0.972981,0.963253,0,0.000000,0.964785
4,0.25,0.5,0.502591,0.685422,2490,2490,0.972130,0.965341,0,0.000000,1.188013
7,0.50,0.5,0.786884,0.765042,2490,2490,0.971607,0.960683,0,0.021841,1.551926
10,0.75,0.5,1.134984,0.874456,2490,2490,0.971494,0.962048,0,0.260529,2.009440
13,1.00,0.5,1.627620,1.129498,2490,2490,0.972132,0.963655,0,0.498122,2.757118


,algorithm,mean,std,n_valid,n_total,expected_utility,realized_utility,n_undefined,lower,upper
0,LDA-exp,1.650181,1.151700,2490,2490,0.972401,0.963896,0,0.498481,2.801881
1,LDA-tru,0.241053,0.723732,2490,2490,0.972981,0.963253,0,0.000000,0.964785
2,OPT,1.968620,1.411075,1849,2490,0.988842,0.980241,641,0.557545,3.379695


SD uses ddof=1 across model/cohort allocations; it is not a confidence interval.


In [8]:
def rgba(color, opacity):
    r, g, b = (int(color[i:i + 2], 16) for i in (1, 3, 5))
    return f"rgba({r},{g},{b},{opacity})"

fig = go.Figure()
series = []
reference_styles = {
    "LDA-exp": ("#7A5195", "dash"),
    "LDA-tru": ("#009E73", "dot"),
    "OPT": ("#242424", "dashdot"),
}
for algorithm, (color, dash) in reference_styles.items():
    row = reference_stats.set_index("algorithm").loc[algorithm]
    series.append({"name": algorithm, "x": [0, 1], "mean": [row["mean"]] * 2,
                   "std": [row["std"]] * 2, "lower": [row["lower"]] * 2, "upper": [row["upper"]] * 2,
                   "valid": [row["n_valid"]] * 2, "total": [row["n_total"]] * 2,
                   "color": color, "dash": dash, "marker": None,
                   "band_opacity": 0.045 if SHOW_REFERENCE_BANDS else 0})
palette = ["#0072B2", "#D55E00", "#CC79A7", "#56B4E9", "#E69F00", "#009E73"]
markers = ["circle", "diamond", "square", "triangle-up", "cross", "triangle-down"]
for i, delta in enumerate(sorted(experiment_stats.delta.unique(), reverse=True)):
    curve = experiment_stats.loc[experiment_stats.delta.eq(delta)].sort_values("beta")
    series.append({"name": f"δ = {delta:g}", "x": curve.beta.tolist(), "mean": curve["mean"].tolist(),
                   "std": curve["std"].tolist(), "lower": curve.lower.tolist(), "upper": curve.upper.tolist(),
                   "valid": curve.n_valid.tolist(), "total": curve.n_total.tolist(),
                   "color": palette[i % len(palette)], "dash": "solid", "marker": markers[i % len(markers)],
                   "band_opacity": 0.16})

# Draw bands first so every mean line remains visible above the shading.
for s in series:
    if not s["band_opacity"]:
        continue
    for bound, fill in (("lower", None), ("upper", "tonexty")):
        fig.add_trace(go.Scatter(
            x=s["x"], y=s[bound], mode="lines", line=dict(width=0),
            fill=fill, fillcolor=rgba(s["color"], s["band_opacity"]),
            name=s["name"], legendgroup=s["name"], showlegend=False, hoverinfo="skip", connectgaps=False,
        ))
for s in series:
    fig.add_trace(go.Scatter(
        x=s["x"], y=s["mean"], name=s["name"], legendgroup=s["name"],
        mode="lines+markers" if s["marker"] else "lines", connectgaps=False,
        line=dict(color=s["color"], width=2.7 if s["marker"] else 2, dash=s["dash"]),
        marker=dict(symbol=s["marker"] or "circle", size=8),
        customdata=np.column_stack([s["std"], s["valid"], s["total"]]),
        hovertemplate=("β: %{x:g}<br>" if s["marker"] else "")
            + "Mean disparity: %{y:.3f}<br>SD: %{customdata[0]:.3f}"
              "<br>Defined allocations: %{customdata[1]:.0f} / %{customdata[2]:.0f}<extra>%{fullData.name}</extra>",
    ))
upper_values = pd.concat([experiment_stats.upper, reference_stats.upper, experiment_stats["mean"], reference_stats["mean"]])
y_max = max(float(upper_values.max()), 0.1) * 1.08
fig.update_layout(
    template="plotly_white", 
    title=dict(text=("Absolute disparity vs. prediction error retained"
                     f"<br><sup>n={config['sample_size']}, k={config['k']} · "
                     f"{len(loaded_runs)} saved β/δ combinations · mean ± 1 SD</sup>"), x=0.06),
    font=dict(family="Arial, sans-serif", size=14),
    margin=dict(l=85, r=35, t=100, b=120),
    xaxis=dict(title="β — Fraction of prediction error retained before rounding", range=[-0.02, 1.02],
               tickvals=[0, 0.25, 0.5, 0.75, 1], showgrid=False),
    yaxis=dict(title="Absolute disparity evaluated using true needs", range=[0, y_max],
               gridcolor="#E5E7EB", zeroline=False),
    legend=dict(orientation="h", x=0.5, xanchor="center", y=-0.18, yanchor="top", groupclick="togglegroup"),
    hovermode="closest",
)
fig.show()


## Grid spacing at full prediction error (β = 1)

This figure uses only the completed β = 1 runs. The x-axis is δ, the need grid spacing;
smaller values represent finer discretization. The curve shows mean absolute disparity,
evaluated using true needs, with ±1 sample standard deviation clipped at zero.
LDA-exp (the δ = 1 baseline), LDA-tru, and OPT remain horizontal references.
`SHOW_REFERENCE_BANDS` controls their shading in both figures.

The dataframe below shows mean absolute disparity and its sample standard deviation for each δ at β = 1.
Values are displayed to four decimal places; `df_beta_one_disparity` retains full precision.


In [ ]:
df_beta_one_disparity = (
    experiment_stats.loc[experiment_stats["beta"].eq(1.0), ["delta", "mean", "std"]]
    .sort_values("delta")
    .rename(columns={
        "mean": "absolute_disparity_mean",
        "std": "absolute_disparity_std",
    })
    .reset_index(drop=True)
)
display(df_beta_one_disparity.round(4))


,delta,absolute_disparity_mean,absolute_disparity_std
0,0.2,1.6307,1.1377
1,0.5,1.6276,1.1295
2,1.0,1.6502,1.1517


In [12]:
beta_one_stats = experiment_stats.loc[experiment_stats.beta.eq(1.0)].sort_values("delta").copy()
fig_beta_one = None
if beta_one_stats.empty:
    print("No completed β = 1 runs are available yet; skipping the delta figure.")
else:
    delta_values = beta_one_stats.delta.tolist()
    delta_min, delta_max = min(delta_values), max(delta_values)
    delta_padding = max((delta_max - delta_min) * 0.025, 0.025)
    delta_range = [max(0, delta_min - delta_padding), delta_max + delta_padding]
    # A single completed grid still gets visible reference lines.
    reference_x = [delta_min, delta_max] if delta_min < delta_max else delta_range
    delta_series = []
    for algorithm, (color, dash) in reference_styles.items():
        row = reference_stats.set_index("algorithm").loc[algorithm]
        delta_series.append({
            "name": algorithm, "x": reference_x, "mean": [row["mean"]] * 2,
            "std": [row["std"]] * 2, "lower": [row["lower"]] * 2, "upper": [row["upper"]] * 2,
            "valid": [row["n_valid"]] * 2, "total": [row["n_total"]] * 2,
            "color": color, "dash": dash, "marker": None,
            "band_opacity": 0.045 if SHOW_REFERENCE_BANDS else 0,
        })
    delta_series.append({
        "name": "β = 1", "x": delta_values, "mean": beta_one_stats["mean"].tolist(),
        "std": beta_one_stats["std"].tolist(), "lower": beta_one_stats.lower.tolist(),
        "upper": beta_one_stats.upper.tolist(), "valid": beta_one_stats.n_valid.tolist(),
        "total": beta_one_stats.n_total.tolist(), "color": "#0072B2", "dash": "solid",
        "marker": "circle", "band_opacity": 0.16,
    })

    fig_beta_one = go.Figure()
    # Draw clipped bands first, then all mean lines above them.
    for s in delta_series:
        if not s["band_opacity"]:
            continue
        for bound, fill in (("lower", None), ("upper", "tonexty")):
            fig_beta_one.add_trace(go.Scatter(
                x=s["x"], y=s[bound], mode="lines", line=dict(width=0),
                fill=fill, fillcolor=rgba(s["color"], s["band_opacity"]),
                name=s["name"], legendgroup=s["name"], showlegend=False, hoverinfo="skip", connectgaps=False,
            ))
    for s in delta_series:
        fig_beta_one.add_trace(go.Scatter(
            x=s["x"], y=s["mean"], name=s["name"], legendgroup=s["name"],
            mode="lines+markers" if s["marker"] else "lines", connectgaps=False,
            line=dict(color=s["color"], width=2.7 if s["marker"] else 2, dash=s["dash"]),
            marker=dict(symbol=s["marker"] or "circle", size=8),
            customdata=np.column_stack([s["std"], s["valid"], s["total"]]),
            hovertemplate=("δ: %{x:g}<br>" if s["marker"] else "")
                + "Mean disparity: %{y:.3f}<br>SD: %{customdata[0]:.3f}"
                  "<br>Defined allocations: %{customdata[1]:.0f} / %{customdata[2]:.0f}<extra>%{fullData.name}</extra>",
        ))
    delta_upper_values = pd.concat([
        beta_one_stats.upper, reference_stats.upper, beta_one_stats["mean"], reference_stats["mean"]])
    fig_beta_one.update_layout(
        template="plotly_white",
        title=dict(text=("Absolute disparity vs. grid spacing"), x=0.06),
        font=dict(family="iosevka", size=14),
        margin=dict(l=85, r=35, t=100, b=120),
        xaxis=dict(title="δ — Need grid spacing (smaller = finer)", range=delta_range,
                   tickvals=delta_values, type="linear", showgrid=False),
        yaxis=dict(title="Absolute disparity",
                   range=[0, max(float(delta_upper_values.max()), 0.1) * 1.08],
                   gridcolor="#E5E7EB", zeroline=False),
        legend=dict(orientation="h", x=0.5, xanchor="center", y=-0.18, yanchor="top", groupclick="togglegroup"),
        hovermode="closest",
    )
    fig_beta_one.show()


In [ ]:
if SAVE_OUTPUTS:
    OUTPUT_DIR = EXPERIMENT_DIR / "analysis"
    OUTPUT_DIR.mkdir(exist_ok=True)
    df_results.to_csv(OUTPUT_DIR / "allocation_metrics.csv", index=False)
    experiment_stats.to_csv(OUTPUT_DIR / "disparity_by_beta_delta.csv", index=False)
    reference_stats.to_csv(OUTPUT_DIR / "disparity_references.csv", index=False)
    pd.DataFrame(reference_sources).to_csv(OUTPUT_DIR / "reference_sources.csv", index=False)
    fig.write_html(OUTPUT_DIR / "absolute_disparity_vs_beta.html", include_plotlyjs=True, full_html=True)
    if fig_beta_one is not None:
        beta_one_stats.to_csv(OUTPUT_DIR / "disparity_vs_delta_beta_1.csv", index=False)
        fig_beta_one.write_html(OUTPUT_DIR / "absolute_disparity_vs_delta_beta_1.html", include_plotlyjs=True, full_html=True)
    print("Saved tables and standalone interactive plots to:", OUTPUT_DIR)
